# Overlays: relation layers and overlapping spans

Examples of relation layers and overlapping spans as the overlay. The outputs are saved as
images (`render_svg_as_img=True`), so they show without running the notebook.

- `overlay_layer` can be a span layer or a **relation layer**. A relation layer gives one
  box per named span of each relation, labelled `name (relation index)`; with
  `overlay_label` the relation attribute is a second line inside the box.
- **Label lanes**: boxes that overlap never share a label line. Shorter boxes get the upper
  lanes, so a box inside another one has its label above the outer box's label, and the
  outer box reaches down around it. Nested and identical boxes are inset, so every edge
  stays visible.
- **Shading**: each box has a semi-transparent fill, so where boxes overlap the fills add
  up and get darker.

In [1]:
from estnltk import Layer, Text
from estnltk_core import RelationLayer

from estnltk_syntax_vis import DependencyTreeVisualiser


def show(text, **config):
    """Draw `text`, saved as an image so the output shows without re-running."""
    DependencyTreeVisualiser(render_svg_as_img=True, **config)(text)

## Span layer: partial and nested named entities

From `edge_cases.ipynb`: "rektor Toomas Asser" is `PER+TITLE` and "Toomas Asser" inside it
is `PER`. Before, both labels were drawn on one line on top of each other, and without
labels the two boxes looked like one.

In [2]:
ner_text = Text(
    "Tartu Ülikooli rektor Toomas Asser ja Eesti Kunstiakadeemia esindaja kohtusid"
).tag_layer(["maltparser_syntax"])
entities = Layer(
    "entities", attributes=("label",), text_object=ner_text, enveloping="words"
)
for start, end, label in [
    (0, 2, "ORG"),
    (2, 5, "PER+TITLE"),
    (3, 5, "PER"),
    (6, 8, "ORG"),
    (9, 10, "EVENT"),
]:
    entities.add_annotation(ner_text.words[start:end], label=label)
ner_text.add_layer(entities)

show(
    ner_text,
    syntax_layer="maltparser_syntax",
    overlay_layer="entities",
    overlay_label="label",
)

## Span layer: clauses (discontinuous and embedded)

The main clause "Mees … oli tuttav ja" is discontinuous; its
`start .. end` range contains the embedded clause, which is therefore drawn inside it.
No special handling of discontinuity is needed.

In [3]:
clause_text = Text("Mees, keda seal kohtasime, oli tuttav ja teretas meid.")
clause_text.tag_layer(["maltparser_syntax", "clauses"])
show(
    clause_text,
    syntax_layer="maltparser_syntax",
    overlay_layer="clauses",
    overlay_label="clause_type",
)

## Relation layer: coreference

From the RelationLayer tutorial. "Mari" is the `entity` of relations 0 and 1, and
"Sipsikut" of relations 2 and 3, so their boxes are stacked. Relation 3's `mention` is in
the second sentence.

In [4]:
coref_text = Text(
    'Mari kirjeldas õhinal, kuidas ta väiksena "Sipsikut" luges: "Ma ei suutnud '
    'seda raamatut kohe kuidagi käest ära panna! Nii põnev oli see!"'
).tag_layer(["maltparser_syntax"])
coref = RelationLayer(
    "coreference", span_names=["mention", "entity"], text_object=coref_text
)
coref.add_annotation(mention=(30, 32), entity=(0, 4))
coref.add_annotation(mention=(61, 63), entity=(0, 4))
coref.add_annotation(mention=(75, 88), entity=(42, 52))
coref.add_annotation(mention=(133, 136), entity=(42, 52))
coref_text.add_layer(coref)

show(coref_text, syntax_layer="maltparser_syntax", overlay_layer="coreference")

## Relation layer: temporal relations, with `overlay_label`

"ära võetud" is `entity_a` of three relations, so three identical boxes are stacked, each
with its `rel_type`.

In [5]:
temporal_text = Text(
    "Pühapäeva varahommikul kutsuti politsei Riia mäele. Seal oli ühelt noorelt "
    "mehelt ära võetud nahktagi ja käekell. Juhtumi kohta algatati uurimine."
).tag_layer(["maltparser_syntax"])
tlinks = RelationLayer(
    "temporal_relations",
    span_names=["entity_a", "entity_b"],
    attributes=["rel_type"],
    text_object=temporal_text,
)
tlinks.add_annotation(entity_a=(0, 22), entity_b=(23, 30), rel_type="INCLUDES")
tlinks.add_annotation(entity_a=(82, 92), entity_b=(23, 30), rel_type="BEFORE")
tlinks.add_annotation(entity_a=(82, 92), entity_b=(114, 121), rel_type="IDENTITY")
tlinks.add_annotation(entity_a=(82, 92), entity_b=(128, 145), rel_type="BEFORE")
temporal_text.add_layer(tlinks)

show(
    temporal_text,
    syntax_layer="maltparser_syntax",
    overlay_layer="temporal_relations",
    overlay_label="rel_type",
)

## Relation layer: a frame, with `overlay_label`

No overlaps, but one relation with six named spans. The relation attribute (`sense_id`) is
repeated in every box of the relation.

In [6]:
frame_text = Text(
    "President Bush kohtus temaga privaatselt Valges Majas teisipäeval."
).tag_layer(["maltparser_syntax"])
frames = RelationLayer(
    "frames",
    span_names=["verb", "arg0", "arg1", "argm_mnr", "argm_tmp", "argm_loc"],
    attributes=["sense_id"],
    text_object=frame_text,
)
frames.add_annotation(
    verb=(15, 21),
    arg0=(0, 14),
    arg1=(22, 28),
    argm_mnr=(29, 40),
    argm_tmp=(54, 65),
    argm_loc=(41, 53),
    sense_id="kohtuma_1",
)
frame_text.add_layer(frames)

show(
    frame_text,
    syntax_layer="maltparser_syntax",
    overlay_layer="frames",
    overlay_label="sense_id",
)